# Chapter 8: Case Studies and Future Directions

This notebook complements Chapter 8 by comparing platform pressures across the book's recurring systems and the public case patterns.

Use it comparatively:
- retail ranking for freshness, feature reuse, and release discipline
- enterprise support assistant for retrieval, prompt governance, context control, and tool traces

The main question is not which stack is universally best. It is which next platform investment best fits the dominant pressure the organization is actually under.

In [ ]:
from pathlib import Path

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
MODULE_PATH = ensure_path_exists(
    COMPANION_ROOT / "code" / "case_studies" / "industry_case_patterns.py",
    "helper module",
)
DATASET_PATH = ensure_path_exists(
    COMPANION_ROOT / "datasets" / "case_studies" / "sample_case_study_patterns.csv",
    "dataset",
)

print("Companion root located successfully.")
print(f"Module path exists: {MODULE_PATH.exists()}")
print(f"Dataset path exists: {DATASET_PATH.exists()}")

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location

import pandas as pd

module_spec = spec_from_file_location('chapter8_case_patterns', MODULE_PATH)
case_module = module_from_spec(module_spec)
assert module_spec.loader is not None
module_spec.loader.exec_module(case_module)

build_case_recommendations = case_module.build_case_recommendations
find_future_pressure_cases = case_module.find_future_pressure_cases
load_case_profiles = case_module.load_case_profiles
summarize_case_landscape = case_module.summarize_case_landscape

pd.set_option('display.max_columns', None)
print('Chapter 8 helper module loaded successfully.')


### Why the Setup Code Matters

The helper functions here are intentionally simple. Their job is to compare pressures across cases, not to simulate an enterprise platform. Read the outputs as a prioritization aid: which pressure dominates, which asset is under stress, and which investment should logically come next.

## Part 1: Load Case Profiles

The scenario table spans the five case-study patterns in the chapter, including the enterprise support assistant with retrieval and prompt governance. Treat each case as a combination of goal, key asset, dominant pressure, failure mode, and engineering lesson.

In [ ]:
cases_df = load_case_profiles(DATASET_PATH)
print(f"Loaded {len(cases_df)} Chapter 8 case profiles from {DATASET_PATH.name}")
cases_df

### What to Notice in the Case Profiles

Each row is a compressed case-study argument: system goal, dominant pressure, platform shape, likely failure mode, and next investment. Read the cases comparatively so you can see why different organizations need different kinds of platform discipline.


## Part 1A: Build a Full Case Worksheet

The enterprise support assistant is the easiest case for connecting all eight chapters into one operating story.

In [ ]:
case_worksheet = pd.DataFrame([
    {'chapter_pressure': 'ingestion', 'diagnosis': 'documents, prompts, and tool traces must all preserve replay boundaries'},
    {'chapter_pressure': 'storage', 'diagnosis': 'corpus snapshots, prompt versions, and traces must remain reconstructable'},
    {'chapter_pressure': 'processing', 'diagnosis': 'chunking, embeddings, retrieval, and prompt assembly change meaning together'},
    {'chapter_pressure': 'governance', 'diagnosis': 'prompt approvals, corpus permissions, and trace retention need named authority'},
    {'chapter_pressure': 'serving', 'diagnosis': 'runtime behavior depends on prompt, retrieval, and tool state as much as on the model'},
    {'chapter_pressure': 'evidence', 'diagnosis': 'a disputed answer needs prompt, chunk, tool, and benchmark evidence to be defensible'},
])
case_worksheet

### Why the Full Worksheet Matters

The case stops being a pattern summary and becomes a full platform review once the chapter pressures are mapped explicitly against one assistant scenario.

## Part 2: Summarize the Chapter 8 Landscape

This summary shows how many cases are dominated by latency, governance, multimodal pressure, agentic workflow state, or strong feature reuse.

In [ ]:
summary_df = summarize_case_landscape(cases_df)
summary_df

### Interpreting the Landscape Summary

The summary is useful when it shows what kind of pressure is becoming common across cases. That helps you distinguish a one-off local problem from a broader shift toward stronger governance, retrieval state management, or workflow-state control.


## Part 3: Build Case Recommendations

The helper adds an emphasis area, a future-pressure signal, and a recommended next move for each case. Compare those outputs against the chapter's synthesis logic: what key asset is under pressure, what failure appears first, and what investment would make that contract more explicit?

In [ ]:
plan_df = build_case_recommendations(cases_df)
plan_df[[
    "case_name",
    "industry",
    "emphasis_area",
    "future_pressure",
    "recommended_guidance",
]]

### Interpreting the Recommendation Output

Treat the recommendation table as a prioritization aid. The key question is whether the suggested next move matches the asset that is actually under the most pressure, rather than the investment that simply sounds most modern.


## Part 3A: Rank the Next Investment Under Constraints

Strategic judgment improves when the platform has less money, fewer people, and more regulatory pressure than the architecture diagram would prefer.

In [ ]:
investment_ranking = pd.DataFrame([
    {'investment': 'retrieval evaluation pipeline', 'budget_fit': 'medium', 'team_fit': 'medium', 'regulatory_fit': 'high', 'why_now': 'answers are only defensible if retrieval quality is measured explicitly'},
    {'investment': 'full agent memory platform', 'budget_fit': 'low', 'team_fit': 'low', 'regulatory_fit': 'medium', 'why_now': 'valuable later, but overbuilt for the current constraint set'},
    {'investment': 'prompt registry and deployment manifest', 'budget_fit': 'high', 'team_fit': 'high', 'regulatory_fit': 'high', 'why_now': 'small artifact cost with immediate control over runtime behavior'},
])
investment_ranking

## Part 3B: Change the Pressure, Watch the Investment Move

The same system can justify a different platform move once governance or agentic pressure changes enough.

In [ ]:
pressure_shift = pd.DataFrame([
    {'scenario_version': 'base assistant', 'governance_pressure': 'high', 'agentic_pressure': 'medium', 'recommended_next_investment': 'prompt registry and deployment manifest'},
    {'scenario_version': 'agentic escalation', 'governance_pressure': 'high', 'agentic_pressure': 'critical', 'recommended_next_investment': 'workflow checkpoints and governed tool traces'},
])
pressure_shift

### Reflective Prompt

Which contract fails first in your chosen case, and what evidence would be missing when the team tries to prove what happened? A good answer should name both the failing contract and the missing artifact or trace.

## Part 4: Identify the Strongest Future-Direction Pressure

These are the cases most exposed to multimodal, governance, retrieval, or agentic workflow pressure.

In [ ]:
future_df = find_future_pressure_cases(plan_df)
future_df

### What the Future-Pressure View Should Tell You

This output should show where the platform is about to outgrow its current operating model. If multimodal, retrieval, or agentic pressures cluster in the same cases, runtime context is becoming a first-class engineering problem rather than an optional extension.


## Part 5: Compare Multimodal and Agentic Cases

This view is most useful when you compare it against the enterprise support-assistant case. Retrieval state, prompt templates, human evaluation traces, and long-running workflow memory all push the platform toward stronger runtime-context engineering rather than a purely classical ML stack.

In [ ]:
next_wave_view = plan_df.loc[
    plan_df["future_pressure"].isin([
        "retrieval and multimodal state pressure is rising",
        "agentic workflow state is becoming a platform concern",
    ]),
    [
        "case_name",
        "industry",
        "multimodal_pressure",
        "agentic_pressure",
        "future_pressure",
        "recommended_guidance",
    ],
]
next_wave_view.sort_values(["industry", "case_name"]).reset_index(drop=True)

### How to Read the Final Comparison

The final comparison matters only if it changes your prioritization logic. For each case, ask:

- What is the dominant pressure?
- What is the key asset under that pressure?
- What failure would appear first if the next investment were deferred?
- Why can another plausible investment wait?

That is the chapter's closing move: turn architecture admiration into platform judgment.

## Part 5A: Inspect the Investment Prioritization Artifact

The chapter names an investment-prioritization artifact because case-study synthesis should translate into a concrete next-move worksheet. Use the table below as a compact version of that decision aid.

In [ ]:
investment_path = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "case_studies" / "investment_prioritization.csv",
    "artifact",
)
investment_df = pd.read_csv(investment_path)
investment_df

## Exercise

### Concept check
- Map multimodal growth, agentic workflows, stronger regulation, retrieval-heavy systems, and platform convergence to the contract each one stresses first.

### Diagnostic scenario
- Choose one case profile and complete a six-contract diagnosis naming the current pressure, first likely failure, and needed investment for each contract.

### Artifact-building exercise
- Write a short platform design memo for the next version of the chosen system. State the dominant pressure, the contract under most stress, the highest-priority investment, the evidence required after failure, and the first rollback action.

### Notebook extension
- Re-rank at least three investments under limited budget, a small team, high governance pressure, and growing retrieval use. Finish with a one-page platform investment recommendation naming what moves first, what can wait, and what fails if the chosen move is delayed.